# Unit 5 — 02: Automated Retraining Pipeline

**What you will do:** Build a full retraining decision loop — detect when to retrain, run retraining, compare old vs new model, and promote the winner.

**Why it matters:** A model deployed today will drift over time. Without automated retraining, you are shipping a product whose quality silently degrades.

**How to run:** Python 3.10+. Run cells in order.

---

## Section 1 — When to Retrain

Three standard triggers for retraining in production:

| Trigger | Condition | Example |
|---|---|---|
| **Scheduled** | Fixed cadence (weekly/monthly) | Every Sunday at 2am regardless of metrics |
| **Performance-based** | Accuracy drops below a threshold | Accuracy < 85% triggers retraining |
| **Drift-detected** | Input distribution has shifted | KS test p-value < 0.05 on feature distributions |

In practice, you combine all three: schedule keeps the model fresh, and drift/performance triggers catch sudden changes.

## Section 2 — Simulate a Drift Scenario

Train on clean iris data. Evaluate on a noisy (shifted) version. Watch accuracy drop.

In [ ]:
import numpy as np
import pathlib
import json
import joblib
from typing import Dict, Tuple, List

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from scipy import stats as sp_stats

rng = np.random.default_rng(42)

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train the initial production model
model_v1 = RandomForestClassifier(n_estimators=50, random_state=42)
model_v1.fit(X_train, y_train)
acc_original = model_v1.score(X_test, y_test)
print(f"Model v1 accuracy on clean test data:   {acc_original:.3f}")

# Simulate drift: add substantial noise to represent new production data
X_test_drifted = X_test + rng.normal(loc=0.5, scale=0.8, size=X_test.shape)
acc_drifted = model_v1.score(X_test_drifted, y_test)
print(f"Model v1 accuracy on drifted test data: {acc_drifted:.3f}")
print(f"Accuracy drop: {acc_original - acc_drifted:.3f}")

## Section 3 — The Retraining Decision Function

`should_retrain()` centralizes all trigger logic. It returns `True` and a human-readable reason, or `False` if no trigger fires.

In [ ]:
def should_retrain(
    current_accuracy: float,
    accuracy_threshold: float,
    drift_score: float,
    drift_threshold: float,
) -> Tuple[bool, str]:
    """
    Returns (True, reason) if retraining is needed, else (False, 'OK').
    drift_score: KS statistic (0-1). Higher = more drift.
    """
    if current_accuracy < accuracy_threshold:
        return True, f"Accuracy {current_accuracy:.3f} < threshold {accuracy_threshold:.3f}"
    if drift_score > drift_threshold:
        return True, f"Drift score {drift_score:.3f} > threshold {drift_threshold:.3f}"
    return False, "OK — no trigger fired"


# Compute a KS drift score on the first feature
ks_stat, ks_pvalue = sp_stats.ks_2samp(X_test[:, 0], X_test_drifted[:, 0])
print(f"KS drift score on feature 0: {ks_stat:.3f}  (p={ks_pvalue:.4f})")

scenarios = [
    (acc_original, 0.85, ks_stat, 0.30, "Normal operation"),
    (acc_drifted,  0.85, ks_stat, 0.30, "Low accuracy"),
    (acc_original, 0.85, 0.40,    0.30, "High drift"),
    (acc_drifted,  0.85, 0.40,    0.30, "Both triggers"),
]

print("\n{:<25} {:<10} {}".format("Scenario", "Retrain?", "Reason"))
print("-" * 70)
for acc, acc_thr, drift, drift_thr, label in scenarios:
    retrain, reason = should_retrain(acc, acc_thr, drift, drift_thr)
    print(f"{label:<25} {str(retrain):<10} {reason}")

## Section 4 — The Retraining Pipeline Function

`retrain_pipeline()` trains a new model, evaluates it, and saves it with a versioned filename.

In [ ]:
MODEL_DIR = pathlib.Path("/tmp/models")
MODEL_DIR.mkdir(exist_ok=True)


def retrain_pipeline(
    new_data: Tuple[np.ndarray, np.ndarray],
    validation_data: Tuple[np.ndarray, np.ndarray],
    version: int = 1,
) -> Tuple[Dict, object]:
    """
    Train a new model on new_data, evaluate on validation_data.
    Save to a versioned file and return (metrics, model).
    """
    X_new, y_new = new_data
    X_val, y_val = validation_data

    new_model = RandomForestClassifier(n_estimators=100, random_state=42)
    new_model.fit(X_new, y_new)

    val_accuracy = float(new_model.score(X_val, y_val))
    model_path = MODEL_DIR / f"model_v{version}.pkl"
    joblib.dump(new_model, model_path)

    metrics = {
        "version": version,
        "val_accuracy": round(val_accuracy, 4),
        "training_samples": len(X_new),
        "model_path": str(model_path),
    }
    print(f"  Saved model v{version} -> {model_path}  (val_acc={val_accuracy:.3f})")
    return metrics, new_model


# Simulate collecting new data from production (with some drift corrected)
X_new_train = X_train + rng.normal(loc=0.2, scale=0.3, size=X_train.shape)
y_new_train = y_train

print("Running retrain pipeline v2...")
metrics_v2, model_v2 = retrain_pipeline(
    new_data=(X_new_train, y_new_train),
    validation_data=(X_test, y_test),
    version=2,
)

## Section 5 — Full Retraining Loop: Detect → Retrain → Compare → Promote

> **Read the numbers carefully.** Below, the champion (v1) accuracy is its score on the *drifted* production data (which is why it looks low), while the challenger (v2) is scored on the *clean* validation set. That makes v2 look dramatically better and it gets promoted. In a real champion/challenger test you must score **both models on the same held-out set** to compare fairly — here we show the simplified version so the promotion logic is easy to follow.

In [ ]:
import matplotlib.pyplot as plt

# Current production state
current_model = model_v1
current_version = 1
# Simulate: the model is evaluated on recent (drifted) production data
current_accuracy = acc_drifted

ACCURACY_THRESHOLD = 0.85
DRIFT_THRESHOLD = 0.30
version_history: List[Dict] = [{"version": 1, "accuracy": acc_drifted, "promoted": True}]

print(f"Production accuracy (on drifted data): {current_accuracy:.3f}")
retrain_needed, reason = should_retrain(current_accuracy, ACCURACY_THRESHOLD, ks_stat, DRIFT_THRESHOLD)
print(f"Retrain? {retrain_needed} — {reason}")

if retrain_needed:
    print("\n--- Starting retraining ---")
    # In practice, new_train combines old training data + recent labeled production data
    X_combined = np.vstack([X_train, X_test_drifted[:20]])
    y_combined = np.concatenate([y_train, y_test[:20]])

    new_version = current_version + 1
    metrics_new, candidate_model = retrain_pipeline(
        new_data=(X_combined, y_combined),
        validation_data=(X_test, y_test),
        version=new_version,
    )
    new_accuracy = metrics_new["val_accuracy"]

    print(f"\n--- Champion vs Challenger ---")
    print(f"  v{current_version} (champion):  {current_accuracy:.3f}")
    print(f"  v{new_version}  (challenger): {new_accuracy:.3f}")

    if new_accuracy > current_accuracy:
        current_model = candidate_model
        current_version = new_version
        current_accuracy = new_accuracy
        version_history.append({"version": new_version, "accuracy": new_accuracy, "promoted": True})
        print(f"  PROMOTED v{new_version} to production.")
    else:
        version_history.append({"version": new_version, "accuracy": new_accuracy, "promoted": False})
        print(f"  REJECTED v{new_version} — challenger did not beat champion.")

# Visualise model version history
versions   = [v["version"]  for v in version_history]
accuracies = [v["accuracy"] for v in version_history]
colors     = ["green" if v["promoted"] else "red" for v in version_history]

plt.figure(figsize=(7, 4))
plt.bar([f"v{v}" for v in versions], accuracies, color=colors, alpha=0.8)
plt.axhline(ACCURACY_THRESHOLD, color="orange", linestyle="--", label="Threshold")
plt.title("Model Version Accuracy (green=promoted, red=rejected)")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig("/tmp/retrain_versions.png", dpi=72)
plt.show()

## Section 6 — Scheduling Retraining

In production you want `check_and_retrain()` to run automatically. Two options:

In [ ]:
# --- Option A: APScheduler (install with: pip install apscheduler) ---
try:
    from apscheduler.schedulers.background import BackgroundScheduler

    def check_and_retrain():
        print("[Scheduler] Running retrain check...")
        # In production: fetch current metrics from monitoring, run should_retrain(), etc.

    scheduler = BackgroundScheduler()
    scheduler.add_job(check_and_retrain, "interval", hours=24)
    # scheduler.start()  # uncomment to actually start the scheduler
    print("APScheduler configured: check_and_retrain() will run every 24 hours.")

except ImportError:
    print("APScheduler not installed. Showing threading.Timer alternative:")

    # --- Option B: threading.Timer (stdlib, no install needed) ---
    import threading

    def schedule_retrain_check(interval_seconds: float = 86400.0):
        """Run check_and_retrain every interval_seconds using a recurring Timer."""
        def _run():
            print("[Timer] Running retrain check...")
            schedule_retrain_check(interval_seconds)  # reschedule

        timer = threading.Timer(interval_seconds, _run)
        timer.daemon = True  # won't block program exit
        # timer.start()  # uncomment to actually start
        return timer

    timer = schedule_retrain_check(interval_seconds=5.0)  # 5s demo; use 86400 in production
    print(f"threading.Timer configured (interval=5s demo). Start with timer.start()")

print("\nProduction cron equivalent (in crontab):")
print("  0 2 * * 0   python /opt/ml/retrain.py   # every Sunday at 2am")

## Summary

The retraining decision loop:

1. **Detect** — one of three triggers fires: scheduled interval, accuracy below threshold, or drift score above threshold
2. **Retrain** — train a challenger model on new data (or new + old data combined)
3. **Compare** — evaluate both champion and challenger on the same held-out validation set
4. **Promote or reject** — only swap to the new model if it strictly beats the current one
5. **Schedule** — run this loop automatically with APScheduler, cron, or Airflow

## Self-Check (answer before scrolling back up)

1. **What is the risk of retraining too often?**  
   Each retraining cycle costs compute and engineering time. More critically, if you retrain on very small windows of new data you may overfit to recent noise and reduce generalization. You can also accidentally retrain on corrupt or anomalous data before noticing the problem.

2. **Why compare new model accuracy to the old model before promoting?**  
   A new model trained on more data is not guaranteed to be better. Retraining could produce a worse model if the new data is noisy, mislabeled, or if the training run diverged. Never replace a working model without a side-by-side comparison.

3. **What data do you train the new model on — just new data, or new + old?**  
   Usually a combination. Pure new data risks forgetting earlier patterns (catastrophic forgetting). Pure old data does not fix the drift. The typical approach is a sliding window: the most recent N weeks of labeled data, which includes both old context and recent distribution.